In [179]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from IPython.display import Markdown, display
from pandas_datareader import data as web
import datetime as dt
import itertools
import statsmodels.api as sm

In [ ]:
# 1.1 Read the data

In [98]:
card = pd.read_excel("/Users/xiaozhuzhang/Desktop/CU 25fall/Credit Risk Analytics/L5/card.xlsx")
cre = pd.read_excel("/Users/xiaozhuzhang/Desktop/CU 25fall/Credit Risk Analytics/L5/CRE.xlsx")

print(card.head())
print(cre.head())

        date      loans  chargeoffs
0 2001-03-31  227610689     2884219
1 2001-06-30  238509983     3159203
2 2001-09-30  230994535     3050252
3 2001-12-31  252496970     3893403
4 2002-03-31  270137409     5172643
        date      loans  chargeoffs
0 2001-03-31  563949186      237847
1 2001-06-30  572472744      233341
2 2001-09-30  589598568      277154
3 2001-12-31  604385934      481878
4 2002-03-31  619573430      340219


In [26]:
# 1.2 calculate charge-off rate

In [99]:
card["date"] = pd.to_datetime(card["date"])
cre["date"]  = pd.to_datetime(cre["date"])
card = card.set_index("date").sort_index()
cre  = cre.set_index("date").sort_index()

card["chargeoff_pct_card"] = card["chargeoffs"] / card["loans"] * 100
cre["chargeoff_pct_cre"]  = cre["chargeoffs"]  / cre["loans"]  * 100

print(card.head())
print(cre.head())

                loans  chargeoffs  chargeoff_pct_card
date                                                 
2001-03-31  227610689     2884219        1.2671720351
2001-06-30  238509983     3159203        1.3245579746
2001-09-30  230994535     3050252        1.3204866513
2001-12-31  252496970     3893403        1.5419602857
2002-03-31  270137409     5172643        1.9148192097
                loans  chargeoffs  chargeoff_pct_cre
date                                                
2001-03-31  563949186      237847       0.0421752537
2001-06-30  572472744      233341       0.0407601938
2001-09-30  589598568      277154       0.0470072376
2001-12-31  604385934      481878       0.0797301811
2002-03-31  619573430      340219       0.0549118125


In [ ]:
#1.2 Stationary Test

In [100]:
result_card = adfuller(card["chargeoff_pct_card"].dropna())
result_cre = adfuller(cre["chargeoff_pct_cre"].dropna())

print(result_card[1])
print(result_cre[1])

display(Markdown(f"charge0ff_pct of card is not stationary"))
display(Markdown(f"charge0ff_pct of cre is not stationary"))

0.05327023403448018
0.4891290849015063


charge0ff_pct of card is not stationary

charge0ff_pct of cre is not stationary

In [ ]:
#1.3 difference

In [101]:
card["chargeoff_pct_card_diff"] = card["chargeoff_pct_card"].diff()
card = card.dropna(subset=["chargeoff_pct_card_diff"])

print(card.head())

cre["chargeoff_pct_cre_diff"] = cre["chargeoff_pct_cre"].diff()
cre = cre.dropna(subset=["chargeoff_pct_cre_diff"])

print(cre.head())

                loans  chargeoffs  chargeoff_pct_card  chargeoff_pct_card_diff
date                                                                          
2001-06-30  238509983     3159203        1.3245579746             0.0573859394
2001-09-30  230994535     3050252        1.3204866513            -0.0040713232
2001-12-31  252496970     3893403        1.5419602857             0.2214736344
2002-03-31  270137409     5172643        1.9148192097             0.3728589240
2002-06-30  275573837     4352632        1.5794794046            -0.3353398050
                loans  chargeoffs  chargeoff_pct_cre  chargeoff_pct_cre_diff
date                                                                        
2001-06-30  572472744      233341       0.0407601938           -0.0014150599
2001-09-30  589598568      277154       0.0470072376            0.0062470438
2001-12-31  604385934      481878       0.0797301811            0.0327229435
2002-03-31  619573430      340219       0.0549118125          

In [102]:
result_card = adfuller(card["chargeoff_pct_card_diff"].dropna())
result_cre = adfuller(cre["chargeoff_pct_cre_diff"].dropna())

print(result_card[1])
print(result_cre[1])

display(Markdown(f"charge0ff_pct_diff of card is stationary"))
display(Markdown(f"charge0ff_pct of cre is stationary"))

0.0015878111917822406
0.013481545256869792


charge0ff_pct_diff of card is stationary

charge0ff_pct of cre is stationary

In [106]:
chargeoff_card = card[["chargeoff_pct_card_diff"]].rename(columns={"chargeoff_pct_card_diff": "chargeoff_card"})
chargeoff_cre  = cre[["chargeoff_pct_cre_diff"]].rename(columns={"chargeoff_pct_cre_diff": "chargeoff_cre "})

chargeoff_pct = chargeoff_card.join(chargeoff_cre, how="inner")

print(chargeoff_pct.head())

            chargeoff_card  chargeoff_cre 
date                                      
2001-06-30    0.0573859394   -0.0014150599
2001-09-30   -0.0040713232    0.0062470438
2001-12-31    0.2214736344    0.0327229435
2002-03-31    0.3728589240   -0.0248183687
2002-06-30   -0.3353398050   -0.0042534366


In [ ]:
#2.1 download data & choose VIXCLS as volatility index

In [82]:
unrate = web.DataReader("UNRATE",        "fred", start="2001-03-31")
oil    = web.DataReader("DCOILBRENTEU",  "fred", start="2001-03-31")
gdp    = web.DataReader("GDP",           "fred", start="2001-03-31")
t10y2y = web.DataReader("T10Y2Y",        "fred", start="2001-03-31")
vix    = web.DataReader("VIXCLS",        "fred", start="2001-03-31")

print(unrate.head())
print(oil.head())
print(gdp.head())
print(t10y2y.head())
print(vix.head())

                 UNRATE
DATE                   
2001-04-01 4.4000000000
2001-05-01 4.3000000000
2001-06-01 4.5000000000
2001-07-01 4.6000000000
2001-08-01 4.9000000000
            DCOILBRENTEU
DATE                    
2001-04-02 23.3100000000
2001-04-03 23.4700000000
2001-04-04 24.3500000000
2001-04-05 24.8700000000
2001-04-06 24.0200000000
                        GDP
DATE                       
2001-04-01 10599.0000000000
2001-07-01 10598.0200000000
2001-10-01 10660.4650000000
2002-01-01 10783.5000000000
2002-04-01 10887.4600000000
                 T10Y2Y
DATE                   
2001-04-02 0.7600000000
2001-04-03 0.7900000000
2001-04-04 0.8200000000
2001-04-05 0.7900000000
2001-04-06 0.8200000000
                  VIXCLS
DATE                    
2001-04-02 31.2100000000
2001-04-03 34.7200000000
2001-04-04 34.0700000000
2001-04-05 29.9400000000
2001-04-06 31.6900000000


In [ ]:
#2.2 Pull the economic data into a pandas data frame

In [110]:
unrate_q = unrate.copy()
unrate_q = unrate.copy()
unrate_q['date'] = [x.date() - dt.timedelta(days=1) for x in unrate_q.index]


unrate_q['year'] = [x.year for x in unrate_q['date']]
unrate_q['month'] = [x.month for x in unrate_q['date']]
unrate_q = unrate_q.loc[unrate_q.index >= "2001-06-30"]
unrate_final = unrate_q[unrate_q['month'].isin([3,6,9,12])]
unrate_final = unrate_final.set_index('date').sort_index()


print(unrate_final.head())

                 UNRATE  year  month
date                                
2001-06-30 4.6000000000  2001      6
2001-09-30 5.3000000000  2001      9
2001-12-31 5.7000000000  2001     12
2002-03-31 5.9000000000  2002      3
2002-06-30 5.8000000000  2002      6


In [92]:
oil_q = oil.copy()
oil_q["date"] = oil_q.index.to_period("Q").to_timestamp(how="end").date
oil_final = oil_q.groupby("date").tail(1).set_index("date").sort_index()

print(oil_final.head())

            DCOILBRENTEU
date                    
2001-06-30 26.2100000000
2001-09-30 21.8700000000
2001-12-31 19.3500000000
2002-03-31 25.3400000000
2002-06-30 25.3300000000


In [ ]:
#2.3 GDP Growth

In [107]:
gdp_q = gdp.copy()
gdp_q["date"] = [x - pd.offsets.Day(1) for x in gdp_q.index]
gdp_q = gdp_q.set_index("date").sort_index()
gdp_q["gdp_lag"] = gdp_q["GDP"].shift(1)
gdp_q["gdp_growth"] = (gdp_q["GDP"] - gdp_q["gdp_lag"]) / gdp_q["gdp_lag"]
gdp_final = gdp_q.iloc[1:]

print(gdp_final.head())

                        GDP          gdp_lag    gdp_growth
date                                                      
2001-06-30 10598.0200000000 10599.0000000000 -0.0000924616
2001-09-30 10660.4650000000 10598.0200000000  0.0058921383
2001-12-31 10783.5000000000 10660.4650000000  0.0115412414
2002-03-31 10887.4600000000 10783.5000000000  0.0096406547
2002-06-30 10984.0400000000 10887.4600000000  0.0088707559


In [94]:
t10y2y_q = t10y2y.copy()
t10y2y_q["date"] = t10y2y_q.index.to_period("Q").to_timestamp(how="end").date
t10y2y_final = t10y2y_q.groupby("date").tail(1).set_index("date").sort_index()

print(t10y2y_final.head())

                 T10Y2Y
date                   
2001-06-30 1.1700000000
2001-09-30 1.7400000000
2001-12-31 2.0000000000
2002-03-31          NaN
2002-06-30 1.9600000000


In [95]:
vix_q = vix.copy()
vix_q["date"] = vix_q.index.to_period("Q").to_timestamp(how="end").date
vix_final = vix_q.groupby("date").tail(1).set_index("date").sort_index()
vix_final = vix_final.rename(columns={"VIXCLS": "VIX"})

print(vix_final.head(10))

                     VIX
date                    
2001-06-30 19.0600000000
2001-09-30 31.9300000000
2001-12-31 23.8000000000
2002-03-31           NaN
2002-06-30 25.4000000000
2002-09-30 39.6900000000
2002-12-31 28.6200000000
2003-03-31 29.1500000000
2003-06-30 19.5200000000
2003-09-30 22.7200000000


In [ ]:
#2.4 combine

In [156]:
for df in [unrate_final, gdp_final, oil_final, t10y2y_final]:
    df.index = pd.to_datetime(df.index)

econ = (
    unrate_final
    .join(gdp_final[["GDP", "gdp_growth"]], how="outer")
    .join(oil_final, how="outer")
    .join(t10y2y_final, how="outer")
).sort_index()

econ = econ.drop(columns=["year", "month"], errors="ignore")
econ = econ.dropna()

print(econ.head())


                 UNRATE              GDP    gdp_growth  DCOILBRENTEU  \
date                                                                   
2001-06-30 4.6000000000 10598.0200000000 -0.0000924616 26.2100000000   
2001-09-30 5.3000000000 10660.4650000000  0.0058921383 21.8700000000   
2001-12-31 5.7000000000 10783.5000000000  0.0115412414 19.3500000000   
2002-06-30 5.8000000000 10984.0400000000  0.0088707559 25.3300000000   
2002-09-30 5.7000000000 11061.4330000000  0.0070459503 29.1100000000   

                 T10Y2Y  
date                     
2001-06-30 1.1700000000  
2001-09-30 1.7400000000  
2001-12-31 2.0000000000  
2002-06-30 1.9600000000  
2002-09-30 1.9100000000  


In [162]:
common_idx = chargeoff_pct.index.intersection(econ.index).sort_values()
combined = chargeoff_pct.loc[common_idx].join(econ.loc[common_idx], how="inner")

print(combined.head())

            chargeoff_card  chargeoff_cre        UNRATE              GDP  \
date                                                                       
2001-06-30    0.0573859394   -0.0014150599 4.6000000000 10598.0200000000   
2001-09-30   -0.0040713232    0.0062470438 5.3000000000 10660.4650000000   
2001-12-31    0.2214736344    0.0327229435 5.7000000000 10783.5000000000   
2002-06-30   -0.3353398050   -0.0042534366 5.8000000000 10984.0400000000   
2002-09-30   -0.1797613545   -0.0001232608 5.7000000000 11061.4330000000   

              gdp_growth  DCOILBRENTEU       T10Y2Y  
date                                                 
2001-06-30 -0.0000924616 26.2100000000 1.1700000000  
2001-09-30  0.0058921383 21.8700000000 1.7400000000  
2001-12-31  0.0115412414 19.3500000000 2.0000000000  
2002-06-30  0.0088707559 25.3300000000 1.9600000000  
2002-09-30  0.0070459503 29.1100000000 1.9100000000  


In [96]:
# 2.5 augmented Dickey-Fuller statistics

In [168]:
result_unrate          = adfuller(combined["UNRATE"].dropna())
result_gdp             = adfuller(combined["GDP"].dropna())
result_gdp_growth      = adfuller(combined["gdp_growth"].dropna())
result_oil             = adfuller(combined["DCOILBRENTEU"].dropna())
result_t10y2y          = adfuller(combined["T10Y2Y"].dropna())
result_chargeoff_card  = adfuller(combined["chargeoff_card"].dropna())
result_chargeoff_cre  = adfuller(combined["chargeoff_cre "].dropna())

print(result_unrate[1])
print(result_gdp[1])
print(result_gdp_growth[1])
print(result_oil[1])
print(result_t10y2y[1])
print(result_chargeoff_card[1])
print(result_chargeoff_cre[1])

0.33296935550148565
0.97775884952529
0.00028498159896295206
0.2167086127036349
0.5930228794431154
0.003956009961040058
0.012370631789911275


In [ ]:
# gdp, GDP_Growth,t10y2y are not stationary

In [171]:
combined["GDP_diff"]   = combined["GDP"].diff()
combined["gdp_growth_diff"] = combined["gdp_growth"].diff()
combined["T10Y2Y_diff"] = combined["T10Y2Y"].diff()


print(combined.head())

            chargeoff_card  chargeoff_cre        UNRATE              GDP  \
date                                                                       
2001-06-30    0.0573859394   -0.0014150599 4.6000000000 10598.0200000000   
2001-09-30   -0.0040713232    0.0062470438 5.3000000000 10660.4650000000   
2001-12-31    0.2214736344    0.0327229435 5.7000000000 10783.5000000000   
2002-06-30   -0.3353398050   -0.0042534366 5.8000000000 10984.0400000000   
2002-09-30   -0.1797613545   -0.0001232608 5.7000000000 11061.4330000000   

              gdp_growth  DCOILBRENTEU       T10Y2Y       GDP_diff  \
date                                                                 
2001-06-30 -0.0000924616 26.2100000000 1.1700000000            NaN   
2001-09-30  0.0058921383 21.8700000000 1.7400000000  62.4450000000   
2001-12-31  0.0115412414 19.3500000000 2.0000000000 123.0350000000   
2002-06-30  0.0088707559 25.3300000000 1.9600000000 200.5400000000   
2002-09-30  0.0070459503 29.1100000000 1.910000

In [175]:
combined_result = combined.iloc[1:]
print(combined_result.head())

            chargeoff_card  chargeoff_cre        UNRATE              GDP  \
date                                                                       
2001-09-30   -0.0040713232    0.0062470438 5.3000000000 10660.4650000000   
2001-12-31    0.2214736344    0.0327229435 5.7000000000 10783.5000000000   
2002-06-30   -0.3353398050   -0.0042534366 5.8000000000 10984.0400000000   
2002-09-30   -0.1797613545   -0.0001232608 5.7000000000 11061.4330000000   
2002-12-31   -0.0361727198    0.0201008385 5.8000000000 11174.1290000000   

             gdp_growth  DCOILBRENTEU       T10Y2Y       GDP_diff  \
date                                                                
2001-09-30 0.0058921383 21.8700000000 1.7400000000  62.4450000000   
2001-12-31 0.0115412414 19.3500000000 2.0000000000 123.0350000000   
2002-06-30 0.0088707559 25.3300000000 1.9600000000 200.5400000000   
2002-09-30 0.0070459503 29.1100000000 1.9100000000  77.3930000000   
2002-12-31 0.0101881917 30.1200000000 2.2200000000 11

In [177]:
combined_result = combined.drop(columns=["GDP", "gdp_growth", "T10Y2Y"], errors="ignore")
combined_result = combined_result.drop(columns=["GDP", "gdp_growth", "T10Y2Y"], errors="ignore")
print(combined_result.head())

            chargeoff_card  chargeoff_cre        UNRATE  DCOILBRENTEU  \
date                                                                    
2001-06-30    0.0573859394   -0.0014150599 4.6000000000 26.2100000000   
2001-09-30   -0.0040713232    0.0062470438 5.3000000000 21.8700000000   
2001-12-31    0.2214736344    0.0327229435 5.7000000000 19.3500000000   
2002-06-30   -0.3353398050   -0.0042534366 5.8000000000 25.3300000000   
2002-09-30   -0.1797613545   -0.0001232608 5.7000000000 29.1100000000   

                 GDP_diff  gdp_growth_diff   T10Y2Y_diff  
date                                                      
2001-06-30            NaN              NaN           NaN  
2001-09-30  62.4450000000     0.0059845999  0.5700000000  
2001-12-31 123.0350000000     0.0056491031  0.2600000000  
2002-06-30 200.5400000000    -0.0026704855 -0.0400000000  
2002-09-30  77.3930000000    -0.0018248056 -0.0500000000  


In [178]:
result_gdp             = adfuller(combined_result["GDP_diff"].dropna())
result_gdp_growth      = adfuller(combined_result["gdp_growth_diff"].dropna())
result_t10y2y          = adfuller(combined_result["T10Y2Y_diff"].dropna())

print(result_gdp[1])
print(result_gdp_growth[1])
print(result_t10y2y[1])

4.9445033319339555e-05
8.71475522567831e-09
1.4105464898020201e-11


In [ ]:
#all stationary

In [ ]:
#3.6 AR1, three-factor model

In [183]:
df = combined_result.copy()

df["card_lag1"] = df["chargeoff_card"].shift(1)
df["cre_lag1"]  = df["chargeoff_cre "].shift(1)
df = df.dropna()

factors = ["UNRATE", "DCOILBRENTEU", "GDP_diff", "gdp_growth_diff", "T10Y2Y_diff"]

factor_combos = list(itertools.combinations(factors, 3))

print(factor_combos)

[('UNRATE', 'DCOILBRENTEU', 'GDP_diff'), ('UNRATE', 'DCOILBRENTEU', 'gdp_growth_diff'), ('UNRATE', 'DCOILBRENTEU', 'T10Y2Y_diff'), ('UNRATE', 'GDP_diff', 'gdp_growth_diff'), ('UNRATE', 'GDP_diff', 'T10Y2Y_diff'), ('UNRATE', 'gdp_growth_diff', 'T10Y2Y_diff'), ('DCOILBRENTEU', 'GDP_diff', 'gdp_growth_diff'), ('DCOILBRENTEU', 'GDP_diff', 'T10Y2Y_diff'), ('DCOILBRENTEU', 'gdp_growth_diff', 'T10Y2Y_diff'), ('GDP_diff', 'gdp_growth_diff', 'T10Y2Y_diff')]


In [ ]:
#3.7 best model(R2)

In [186]:
results_card = []
results_cre = []

for combo in factor_combos:
    # CARD
    X_card = df[list(combo) + ["card_lag1"]]
    X_card = sm.add_constant(X_card)
    y_card = df["chargeoff_card"]
    model_card = sm.OLS(y_card, X_card).fit()
    results_card.append({"Factors": combo, "R2_CARD": model_card.rsquared})

    # CRE
    X_cre = df[list(combo) + ["cre_lag1"]]
    X_cre = sm.add_constant(X_cre)
    y_cre = df["chargeoff_cre "]
    model_cre = sm.OLS(y_cre, X_cre).fit()
    results_cre.append({"Factors": combo, "R2_CRE": model_cre.rsquared})

r2_table = pd.DataFrame(results_card).merge(pd.DataFrame(results_cre), on="Factors")
r2_table = r2_table.sort_values(by=["R2_CARD", "R2_CRE"], ascending=False).reset_index(drop=True)

print(r2_table)

                                        Factors      R2_CARD       R2_CRE
0           (UNRATE, GDP_diff, gdp_growth_diff) 0.2331880651 0.2632821393
1      (GDP_diff, gdp_growth_diff, T10Y2Y_diff) 0.2272727461 0.2810387261
2     (DCOILBRENTEU, GDP_diff, gdp_growth_diff) 0.2100931227 0.3049605515
3        (UNRATE, gdp_growth_diff, T10Y2Y_diff) 0.1874444735 0.2330390491
4  (DCOILBRENTEU, gdp_growth_diff, T10Y2Y_diff) 0.1852774360 0.2933227382
5       (UNRATE, DCOILBRENTEU, gdp_growth_diff) 0.1493814704 0.2704338129
6           (UNRATE, DCOILBRENTEU, T10Y2Y_diff) 0.0454028700 0.2880952403
7         (DCOILBRENTEU, GDP_diff, T10Y2Y_diff) 0.0438338773 0.3320578980
8              (UNRATE, DCOILBRENTEU, GDP_diff) 0.0274546277 0.3058469904
9               (UNRATE, GDP_diff, T10Y2Y_diff) 0.0251394559 0.2824856361


In [ ]:
#3.7 choose the best mode;
#the larger R2, the better model
#For credit card charge-offs, the model with UNRATE, GDP_diff, and gdp_growth_diff achieved the highest R2(≈0.23), indicating that unemployment and GDP growth shifts play a leading role in consumer credit performance.

#For CRE charge-offs, the best combination — DCOILBRENTEU, GDP_diff, and T10Y2Y_diff — yielded the highest R2(≈0.33). This suggests that commercial real estate defaults are more influenced by macroeconomic conditions such as oil price fluctuations, economic activity, and changes in the yield curve.